In [ ]:
import pandas as pd
import numpy as np

# 1. Load filtered temperature dataset (Bogotá and Antioquia)
input_path = r"C:\Users\Germán Calderón\Downloads\ideam_bogota_antioquia.csv"
df = pd.read_csv(input_path, low_memory=False)

# 2. Convert FechaObservacion to datetime
df['date'] = pd.to_datetime(df['FechaObservacion'], format='mixed', errors='coerce')

# 3. Clean and standardize Department column
df['department'] = df['Departamento'].astype(str).str.upper().str.strip()
df['department'] = df['department'].replace({
    'BOGOTA D.C.': 'BOGOTÁ',
    'BOGOTA': 'BOGOTÁ'
})

# 4. Convert ValorObservado to float
df['observed_value'] = (
    df['ValorObservado']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
df['observed_value'] = pd.to_numeric(df['observed_value'], errors='coerce')

# -----------------------------------------------------------------------------
# ANTIOQUIA CORRECTION: Treat false zero values / sensor glitches as NaN
# In Antioquia, measurements <= 0.0 °C are sensor errors / missing data.
# -----------------------------------------------------------------------------
antioquia_zero_filter = (df['department'] == 'ANTIOQUIA') & (df['observed_value'] <= 0.0)
df.loc[antioquia_zero_filter, 'observed_value'] = np.nan

# Drop records with missing dates or temperatures
df = df.dropna(subset=['observed_value', 'date']).copy()

# 5. Extract Year, Month, and Daily Date
df['year'] = df['date'].dt.year.astype(int)
df['month'] = df['date'].dt.month.astype(int)
df['daily_date'] = df['date'].dt.date

# 6. Daily frost evaluation on cleaned data (minimum daily temperature)
df_daily = df.groupby(['year', 'month', 'department', 'daily_date'])['observed_value'].min().reset_index()
df_daily['frost_under_2'] = df_daily['observed_value'] < 2.0
df_daily['frost_under_0'] = df_daily['observed_value'] <= 0.0

# 7. Aggregate metrics monthly
df_monthly_temp = df.groupby(['year', 'month', 'department']).agg(
    avg_temperature=('observed_value', 'mean'),
    min_temperature=('observed_value', 'min'),
    max_temperature=('observed_value', 'max')
).reset_index()

df_monthly_frost = df_daily.groupby(['year', 'month', 'department']).agg(
    frost_days_under_2=('frost_under_2', 'sum'),
    frost_days_under_0=('frost_under_0', 'sum')
).reset_index()

# 8. Merge results into final summary dataframe
df_monthly_summary = pd.merge(
    df_monthly_temp,
    df_monthly_frost,
    on=['year', 'month', 'department']
)

# 9. Round temperature metrics
df_monthly_summary['avg_temperature'] = df_monthly_summary['avg_temperature'].round(2)
df_monthly_summary['min_temperature'] = df_monthly_summary['min_temperature'].round(2)
df_monthly_summary['max_temperature'] = df_monthly_summary['max_temperature'].round(2)

# 10. Reorder columns and save to CSV
final_columns = [
    'year',
    'month',
    'department',
    'avg_temperature',
    'min_temperature',
    'max_temperature',
    'frost_days_under_2',
    'frost_days_under_0'
]

df_monthly_summary = df_monthly_summary[final_columns].sort_values(by=['year', 'month', 'department'])

output_path = r"C:\Users\Germán Calderón\Downloads\monthly_weather_summary.csv"
df_monthly_summary.to_csv(output_path, index=False, encoding='utf-8')

print("Process completed! Monthly weather summary saved to:", output_path)
df_monthly_summary.head(10)